# Offside Detection — Legacy v1 Feature Extraction (Kit-Guided KMeans)
## `01_feature_extraction.ipynb`


In [ ]:
!pip install -q ultralytics

import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Offside_Detector')
DATASET_DIR = BASE_DIR / 'dataset'
OFFSIDE_DIR, OFFSIDE_LEFT_DIR = DATASET_DIR / 'offside', DATASET_DIR / 'offside' / 'left'
ONSIDE_DIR, ONSIDE_LEFT_DIR = DATASET_DIR / 'onside', DATASET_DIR / 'onside' / 'left'
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def count_images_in_folder(folder_path, recursive=False):
    if not folder_path.exists():
        return 0
    files = folder_path.rglob('*') if recursive else folder_path.glob('*')
    return sum(1 for f in files if f.is_file() and f.suffix.lower() in VALID_EXTS)


print(f"Dataset exists: {DATASET_DIR.exists()} | Offside: {count_images_in_folder(OFFSIDE_DIR, True)} | Onside: {count_images_in_folder(ONSIDE_DIR, True)}")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def load_and_align_image(image_path, is_left=False):
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Failed to read image: {image_path}")
    was_flipped = False
    if is_left:
        img = cv2.flip(img, 1)  # Normalize attacks left-to-right
        was_flipped = True
    return img, was_flipped


def get_pitch_mask(img):
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_green = np.array([30, 40, 40], dtype=np.uint8)
    upper_green = np.array([85, 255, 255], dtype=np.uint8)
    raw_mask = cv2.inRange(hsv, lower_green, upper_green)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    closed_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    pitch_mask = np.zeros((h, w), dtype=np.uint8)
    pitch_hull = None
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        pitch_hull = cv2.convexHull(largest_contour)
        cv2.drawContours(pitch_mask, [pitch_hull], -1, 255, thickness=cv2.FILLED)
    else:
        pitch_mask[int(h * 0.2):, :] = 255

    # Trim top ad boards without eroding sides
    erode_kernel = np.ones((45, 15), dtype=np.uint8)
    pitch_mask = cv2.erode(pitch_mask, erode_kernel, iterations=1)
    return pitch_mask, pitch_hull


sample_folder = OFFSIDE_LEFT_DIR if OFFSIDE_LEFT_DIR.exists() and count_images_in_folder(OFFSIDE_LEFT_DIR) > 0 else OFFSIDE_DIR
sample_images = [f for f in sample_folder.glob('*') if f.suffix.lower() in VALID_EXTS]

if sample_images:
    sample_path = sample_images[0]
    is_left_sample = 'left' in str(sample_path).lower()
    aligned_img, was_flipped = load_and_align_image(sample_path, is_left=is_left_sample)
    mask, hull = get_pitch_mask(aligned_img)
    pitch_only = cv2.bitwise_and(aligned_img, aligned_img, mask=mask)
    print(f"Shape: {aligned_img.shape} | Flipped: {was_flipped}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(aligned_img, cv2.COLOR_BGR2RGB)); axes[0].set_title(sample_path.name); axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title("Pitch Mask (Erode 45x15)"); axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(pitch_only, cv2.COLOR_BGR2RGB)); axes[2].set_title("Pitch Isolated"); axes[2].axis('off')
    plt.tight_layout(); plt.show()


In [ ]:
from ultralytics import YOLO

det_model = YOLO('yolov8x.pt')
pose_model = YOLO('yolov8n-pose.pt')

LEGAL_KEYPOINT_INDICES = [0, 1, 2, 3, 4, 5, 6, 11, 12, 13, 14, 15, 16]  # Exclude arms per FIFA Law 11
EXCLUDED_ARM_INDICES = [7, 8, 9, 10]
MIN_CROP_HEIGHT = 120


def detect_players_and_ball(img, pitch_mask, det_model, pose_model, imgsz=1280):
    h, w = img.shape[:2]
    det_results = det_model(img, classes=[0, 32], conf=0.10, imgsz=imgsz, verbose=False)[0]
    person_boxes, ball_pos = [], None

    if det_results.boxes is not None and len(det_results.boxes) > 0:
        boxes_data = det_results.boxes.xyxy.cpu().numpy()
        classes_data = det_results.boxes.cls.cpu().numpy().astype(int)
        confs_data = det_results.boxes.conf.cpu().numpy()

        ball_candidates = []
        for i, (b, cls_id) in enumerate(zip(boxes_data, classes_data)):
            if cls_id == 0:
                person_boxes.append(b)
            elif cls_id == 32:
                ball_conf = confs_data[i]
                if ball_conf < 0.28:  # Reject white socks false positives
                    continue
                bx, by = int((b[0] + b[2]) / 2.0), int((b[1] + b[3]) / 2.0)
                bw, bh = b[2] - b[0], b[3] - b[1]
                if not (0 <= bx < w and 0 <= by < h and pitch_mask[by, bx] > 0):
                    continue
                aspect_ratio = bw / max(bh, 1e-5)
                if aspect_ratio < 0.6 or aspect_ratio > 1.6 or max(bw, bh) > 45:
                    continue
                ball_candidates.append((ball_conf, np.array([float(bx), float(by)], dtype=np.float32)))

        if ball_candidates:
            ball_candidates.sort(key=lambda x: x[0], reverse=True)
            ball_pos = ball_candidates[0][1]

    on_pitch_boxes = []
    for box in person_boxes:
        center_x = int(np.clip((box[0] + box[2]) / 2.0, 0, w - 1))
        foot_y = int(np.clip(box[3] - 2, 0, h - 1))
        if pitch_mask[foot_y, center_x] > 0:  # Foot-only test rejects sidelines
            on_pitch_boxes.append(box)

    players, pose_success, pose_fallback = [], 0, 0
    for box in on_pitch_boxes:
        x1, y1, x2, y2 = box.astype(int)
        bw, bh = x2 - x1, y2 - y1
        cx1, cy1 = max(0, x1 - int(0.10 * bw)), max(0, y1 - int(0.10 * bh))
        cx2, cy2 = min(w, x2 + int(0.10 * bw)), min(h, y2 + int(0.10 * bh))
        crop = img[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue

        pose_conf = 0.15
        if crop.shape[0] < MIN_CROP_HEIGHT:  # Upscale far players for Pose
            scale = 3 if crop.shape[0] < 60 else 2
            crop = cv2.resize(crop, (crop.shape[1] * scale, crop.shape[0] * scale), interpolation=cv2.INTER_LINEAR)
            pose_conf = 0.10

        pose_res = pose_model(crop, conf=pose_conf, verbose=False)[0]
        got_keypoints = (
            pose_res.keypoints is not None
            and len(pose_res.keypoints) > 0
            and pose_res.keypoints.conf is not None
        )

        if got_keypoints:
            confs_all = pose_res.keypoints.conf.cpu().numpy()
            best_k = int(np.argmax(confs_all.mean(axis=1)))
            sx, sy = (cx2 - cx1) / crop.shape[1], (cy2 - cy1) / crop.shape[0]
            conf = confs_all[best_k]
            global_xy = pose_res.keypoints.xy.cpu().numpy()[best_k] * np.array([sx, sy]) + np.array([cx1, cy1])

            legal_pts = [global_xy[idx] for idx in LEGAL_KEYPOINT_INDICES if conf[idx] > 0.25]
            if legal_pts:
                max_point = max(legal_pts, key=lambda pt: pt[0])
                max_x = float(max_point[0])
            else:
                max_x = float(x2)
                max_point = np.array([float(x2), (y1 + y2) / 2.0])

            players.append({
                'box': box, 'keypoints_xy': global_xy, 'keypoints_conf': conf,
                'max_x': max_x, 'max_point': max_point, 'box_height': float(bh),
                'box_width': float(bw), 'center_x': (x1 + x2) / 2.0,
                'center_y': (y1 + y2) / 2.0, 'pose_source': 'keypoints',
            })
            pose_success += 1
        else:
            max_x = float(box[2]) - 0.10 * float(bw)
            players.append({
                'box': box, 'keypoints_xy': None, 'keypoints_conf': None,
                'max_x': max_x, 'max_point': np.array([max_x, (y1 + y2) / 2.0]),
                'box_height': float(bh), 'box_width': float(bw),
                'center_x': (x1 + x2) / 2.0, 'center_y': (y1 + y2) / 2.0,
                'pose_source': 'bbox_fallback',
            })
            pose_fallback += 1

    print(f"Det: {len(person_boxes)} | OnPitch: {len(on_pitch_boxes)} | Pose: {pose_success} | Fallback: {pose_fallback} | Ball: {ball_pos is not None}")
    return players, ball_pos


if 'aligned_img' in locals() and 'mask' in locals():
    detected_players, detected_ball = detect_players_and_ball(aligned_img, mask, det_model, pose_model)
    vis_img = aligned_img.copy()
    for p in detected_players:
        b = p['box'].astype(int)
        box_color = (0, 200, 255) if p['pose_source'] == 'bbox_fallback' else (255, 200, 0)
        cv2.rectangle(vis_img, (b[0], b[1]), (b[2], b[3]), box_color, 2)
        if p['keypoints_xy'] is not None:
            for idx in LEGAL_KEYPOINT_INDICES:
                if p['keypoints_conf'][idx] > 0.25:
                    cv2.circle(vis_img, tuple(p['keypoints_xy'][idx].astype(int)), 4, (0, 255, 0), -1)
            for idx in EXCLUDED_ARM_INDICES:
                if p['keypoints_conf'][idx] > 0.25:
                    cv2.circle(vis_img, tuple(p['keypoints_xy'][idx].astype(int)), 4, (0, 0, 255), -1)
        cv2.circle(vis_img, tuple(p['max_point'].astype(int)), 7, (0, 255, 255), -1)

    if detected_ball is not None:
        cv2.circle(vis_img, (int(detected_ball[0]), int(detected_ball[1])), 8, (255, 0, 255), -1)
        cv2.putText(vis_img, "Ball", (int(detected_ball[0]) - 15, int(detected_ball[1]) - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2)

    plt.figure(figsize=(15, 9)); plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Top-Down Detection | Total: {len(detected_players)} | Ball: {'Yes' if detected_ball is not None else 'No'}")
    plt.axis('off'); plt.show()


In [ ]:
from sklearn.cluster import KMeans

GRASS_GREEN_LOWER = np.array([36, 40, 40], dtype=np.uint8)  # Preserve yellow kits
GRASS_GREEN_UPPER = np.array([85, 255, 255], dtype=np.uint8)
GRASS_RATIO_THRESHOLD = 0.70

COLOR_WEIGHTS = np.array([0.55, 1.4, 1.4], dtype=np.float32)  # Balance white vs dark shading
RESCUE_THRESHOLD = 42.0
GK_MIN_X_RATIO = 0.75
GK_MIN_Y_RATIO = 0.10
GK_MAX_Y_RATIO = 0.92
GK_MIN_ATT_DIST = 55.0
MIN_TEAM_COLOR_SEP = 25.0

KIT_RGB = {
    'white': (235, 235, 235), 'black': (25, 25, 25), 'red': (200, 30, 40),
    'blue': (30, 50, 140), 'navy': (32, 48, 115), 'sky': (110, 180, 230),
    'yellow': (240, 215, 40), 'orange': (240, 130, 30), 'green': (40, 150, 70),
    'pink': (240, 150, 190), 'purple': (110, 50, 140), 'claret': (110, 25, 50),
    'maroon': (120, 20, 35), 'grey': (130, 130, 130), 'gray': (130, 130, 130),
}


def get_jersey_lab_color(img, player):
    h_img, w_img = img.shape[:2]
    x1, y1, x2, y2 = player['box'].astype(int)
    bh, bw = y2 - y1, x2 - x1

    torso_crop_done = False
    if player['keypoints_xy'] is not None and player['keypoints_conf'] is not None:
        kp, kc = player['keypoints_xy'], player['keypoints_conf']
        valid_pts = [kp[idx] for idx in [5, 6, 11, 12] if kc[idx] > 0.25]
        if len(valid_pts) >= 2:
            cx, cy = float(np.mean([pt[0] for pt in valid_pts])), float(np.mean([pt[1] for pt in valid_pts]))
            hs = int(np.clip(0.10 * bh, 6, 30))
            crop_x1, crop_x2 = max(0, int(cx - hs)), min(w_img, int(cx + hs))
            crop_y1, crop_y2 = max(0, int(cy - hs)), min(h_img, int(cy + hs))
            if crop_x2 > crop_x1 and crop_y2 > crop_y1:
                torso_crop_done = True

    if not torso_crop_done:
        crop_y1, crop_y2 = max(0, y1 + int(0.20 * bh)), min(h_img, y1 + int(0.45 * bh))
        crop_x1, crop_x2 = max(0, x1 + int(0.30 * bw)), min(w_img, x1 + int(0.70 * bw))

    jersey_crop = img[crop_y1:crop_y2, crop_x1:crop_x2]
    if jersey_crop.size == 0:
        return np.array([50.0, 0.0, 0.0], dtype=np.float32)

    hsv_crop = cv2.cvtColor(jersey_crop, cv2.COLOR_BGR2HSV)
    green_mask = cv2.inRange(hsv_crop, GRASS_GREEN_LOWER, GRASS_GREEN_UPPER)
    total_px = max(jersey_crop.shape[0] * jersey_crop.shape[1], 1)
    not_green = cv2.bitwise_not(green_mask) if (green_mask.sum() / 255.0 / total_px < GRASS_RATIO_THRESHOLD) else None
    if not_green is not None and not_green.sum() / 255 < 4:
        not_green = None

    lab_crop = cv2.cvtColor(jersey_crop, cv2.COLOR_BGR2Lab)
    if not_green is not None and (not_green > 0).sum() > 0:
        return np.median(lab_crop[not_green > 0], axis=0).astype(np.float32)
    return np.median(lab_crop.reshape(-1, 3), axis=0).astype(np.float32)


def kit_to_weighted_lab(kit):
    rgb = KIT_RGB.get(kit.lower(), None) if isinstance(kit, str) else kit
    if rgb is None:
        return None
    bgr = np.uint8([[[rgb[2], rgb[1], rgb[0]]]])
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab)[0, 0].astype(np.float32) * COLOR_WEIGHTS


def cluster_teams(players, img, attacking_kit=None):
    n, (h_img, w_img) = len(players), img.shape[:2]
    if n < 2:
        return players, [], None, {}

    raw_colors = np.array([get_jersey_lab_color(img, p) for p in players], dtype=np.float32)
    weighted_colors = raw_colors * COLOR_WEIGHTS
    labels = KMeans(n_clusters=min(4, n), random_state=42, n_init='auto').fit_predict(weighted_colors)

    unique_labels, counts = np.unique(labels, return_counts=True)
    count_map = {int(l): int(c) for l, c in zip(unique_labels, counts)}
    center_map = {int(l): weighted_colors[labels == l].mean(axis=0) for l in unique_labels}
    pool = [int(l) for l in unique_labels if count_map[int(l)] >= 2]
    if len(pool) < 2:
        pool = [int(l) for l in unique_labels]

    kit_lab = kit_to_weighted_lab(attacking_kit) if attacking_kit is not None else None
    all_labels = [int(l) for l in unique_labels]

    if kit_lab is not None and len(all_labels) >= 2:
        best_pool = min(pool, key=lambda l: float(np.linalg.norm(center_map[l] - kit_lab)))
        best_all = min(all_labels, key=lambda l: float(np.linalg.norm(center_map[l] - kit_lab)))
        if float(np.linalg.norm(center_map[best_pool] - kit_lab)) > 65.0 and float(np.linalg.norm(center_map[best_all] - kit_lab)) < 55.0:
            att_label = best_all
        else:
            att_label = best_pool
        rem = [l for l in pool if l != att_label] or [l for l in all_labels if l != att_label]
        distinct_rem = [l for l in rem if float(np.linalg.norm(center_map[l] - center_map[att_label])) > MIN_TEAM_COLOR_SEP]
        def_label = max(distinct_rem or rem, key=lambda l: count_map[l])
        chosen_labels = (att_label, def_label)
    else:
        sorted_pool = sorted(pool, key=lambda l: -count_map[l])
        lbl_a, rem = sorted_pool[0], sorted_pool[1:]
        distinct_rem = [l for l in rem if float(np.linalg.norm(center_map[l] - center_map[lbl_a])) > MIN_TEAM_COLOR_SEP]
        chosen_labels = (lbl_a, (distinct_rem or rem)[0]) if rem else (lbl_a,)

    l0 = chosen_labels[0]
    l1 = chosen_labels[1] if len(chosen_labels) > 1 else None
    idx_0 = [i for i in range(n) if labels[i] == l0]
    idx_1 = [i for i in range(n) if l1 is not None and labels[i] == l1]
    c0, c1 = center_map[l0], (center_map[l1] if l1 is not None else center_map[l0])

    excluded_idx = []
    for i in [i for i in range(n) if labels[i] not in chosen_labels]:
        d0, d1 = float(np.linalg.norm(weighted_colors[i] - c0)), float(np.linalg.norm(weighted_colors[i] - c1))
        if min(d0, d1) < RESCUE_THRESHOLD:
            (idx_0 if d0 <= d1 else idx_1).append(i)
        else:
            excluded_idx.append(i)

    if kit_lab is not None and l1 is not None:
        att_idx, def_idx, att_c, def_c = idx_0, idx_1, c0, c1
    else:
        p75_0 = np.percentile([players[i]['max_x'] for i in idx_0], 75) if idx_0 else 0
        p75_1 = np.percentile([players[i]['max_x'] for i in idx_1], 75) if idx_1 else 0
        if p75_0 >= p75_1:
            def_idx, att_idx, def_c, att_c = idx_0, idx_1, c0, c1
        else:
            att_idx, def_idx, att_c, def_c = idx_0, idx_1, c0, c1

    for i in range(n):
        players[i]['team_dist'] = min(float(np.linalg.norm(weighted_colors[i] - att_c)), float(np.linalg.norm(weighted_colors[i] - def_c)))

    gk_candidates = []
    for i in excluded_idx:
        d_att = float(np.linalg.norm(weighted_colors[i] - att_c))
        d_kit = float(np.linalg.norm(weighted_colors[i] - kit_lab)) if kit_lab is not None else 999.0
        if (players[i]['max_x'] / w_img >= GK_MIN_X_RATIO and GK_MIN_Y_RATIO <= players[i]['center_y'] / h_img <= GK_MAX_Y_RATIO
                and d_att >= GK_MIN_ATT_DIST and d_kit >= GK_MIN_ATT_DIST):
            gk_candidates.append((players[i]['max_x'], i))

    gk_idx = max(gk_candidates)[1] if gk_candidates else None
    if gk_idx is not None:
        excluded_idx = [i for i in excluded_idx if i != gk_idx]

    return ([players[i] for i in att_idx], [players[i] for i in def_idx],
            players[gk_idx] if gk_idx is not None else None,
            {'excluded': [players[i] for i in excluded_idx]})


In [ ]:
def get_defensive_line_x(defenders, goalkeeper):
    opp = list(defenders) + ([goalkeeper] if goalkeeper is not None else [])
    xs = sorted([p['max_x'] for p in opp], reverse=True)
    return xs[1] if (goalkeeper is not None and len(xs) >= 2) else xs[0]


def run_full_pipeline(image_path, is_left, det_model, pose_model, label='', attacking_kit=None):
    img, was_flipped = load_and_align_image(image_path, is_left=is_left)
    pitch_mask, _ = get_pitch_mask(img)
    players, ball_pos = detect_players_and_ball(img, pitch_mask, det_model, pose_model)
    if len(players) < 4:
        return None, {}
    attackers, defenders, goalkeeper, _ = cluster_teams(players, img, attacking_kit=attacking_kit)
    if not attackers or not defenders:
        return None, {}

    max_att_x = max(p['max_x'] for p in attackers)
    max_def_x = get_defensive_line_x(defenders, goalkeeper)
    sorted_att = sorted(attackers, key=lambda p: p['max_x'], reverse=True)
    second_att_x = sorted_att[1]['max_x'] if len(sorted_att) > 1 else max_att_x
    return img, {'label': label, 'max_att_x': max_att_x, 'max_def_x': max_def_x, 'second_att_x': second_att_x,
                 'ball_detected': ball_pos is not None, 'predicted_offside': max_att_x > max_def_x, 'flipped': was_flipped}


In [ ]:
import csv
import ipywidgets as widgets
from IPython.display import display, clear_output


def list_dataset():
    items = []
    for folder, is_left, y in [(OFFSIDE_DIR, False, 1), (OFFSIDE_LEFT_DIR, True, 1),
                               (ONSIDE_DIR, False, 0), (ONSIDE_LEFT_DIR, True, 0)]:
        if not folder.exists():
            continue
        for f in sorted(folder.iterdir()):
            if f.is_file() and f.suffix.lower() in VALID_EXTS:
                items.append({'id': str(f.relative_to(DATASET_DIR)), 'path': f, 'is_left': is_left, 'y': y})
    return items


def load_labels(path):
    labels = {}
    if path.exists():
        with open(path, encoding='utf-8') as f:
            for row in csv.DictReader(f):
                if row.get('kit'):
                    labels[row['image_id']] = row['kit']
    return labels


def save_labels(path, labels):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f); w.writerow(['image_id', 'kit'])
        for k, v in labels.items():
            w.writerow([k, v])


items = list_dataset()
my_csv = BASE_DIR / 'attacking_kits.csv'
my_labels = load_labels(my_csv)
todo_indices = [idx for idx, it in enumerate(items) if it['id'] not in my_labels]


class KitAnnotator:
    def __init__(self, items, todo_indices, my_labels, csv_path):
        self.items, self.todo_indices, self.my_labels, self.csv_path = items, todo_indices, my_labels, csv_path
        self.current_pos, self.history = 0, []
        self.colors_config = [
            ('Red', 'red', '#d32f2f', 'white'), ('Blue', 'blue', '#1976d2', 'white'),
            ('White', 'white', '#f5f5f5', 'black'), ('Yellow', 'yellow', '#fbc02d', 'black'),
            ('Black', 'black', '#212121', 'white'), ('Sky', 'sky', '#4fc3f7', 'black'),
            ('Navy', 'navy', '#0d47a1', 'white'), ('Green', 'green', '#388e3c', 'white'),
            ('Pink', 'pink', '#f06292', 'white'), ('Orange', 'orange', '#f57c00', 'white'),
        ]
        self.out_image, self.out_status = widgets.Output(), widgets.Output()
        color_buttons = []
        for label, val, bg, fg in self.colors_config:
            btn = widgets.Button(description=label, layout=widgets.Layout(width='120px', height='36px', margin='3px'),
                                 style={'button_color': bg, 'text_color': fg, 'font_weight': 'bold'})
            btn.on_click(self.make_color_handler(val))
            color_buttons.append(btn)

        self.btn_skip = widgets.Button(description='Skip', layout=widgets.Layout(width='120px', margin='4px'))
        self.btn_undo = widgets.Button(description='Undo', layout=widgets.Layout(width='120px', margin='4px'))
        self.btn_quit = widgets.Button(description='Finish & Save', layout=widgets.Layout(width='120px', margin='4px'), button_style='warning')
        self.btn_skip.on_click(self.on_skip); self.btn_undo.on_click(self.on_undo); self.btn_quit.on_click(self.on_quit)
        self.custom_input = widgets.Text(placeholder='Custom color...', layout=widgets.Layout(width='220px', margin='4px'))
        self.custom_input.on_submit(self.on_custom_submit)

        self.main_box = widgets.VBox([
            self.out_status, self.out_image,
            widgets.HBox(color_buttons[:5]), widgets.HBox(color_buttons[5:]),
            widgets.HBox([self.btn_undo, self.btn_skip, self.btn_quit, self.custom_input]),
        ])
        display(self.main_box); self.show_current()

    def make_color_handler(self, val):
        return lambda b: self.assign_color(val)

    def on_custom_submit(self, sender):
        val = self.custom_input.value.strip().lower()
        if val:
            self.assign_color(val); self.custom_input.value = ''

    def assign_color(self, color_name):
        if self.current_pos >= len(self.todo_indices):
            return
        item = self.items[self.todo_indices[self.current_pos]]
        self.my_labels[item['id']] = color_name
        save_labels(self.csv_path, self.my_labels)
        self.history.append(self.current_pos); self.current_pos += 1; self.show_current()

    def on_skip(self, b):
        if self.current_pos < len(self.todo_indices):
            self.current_pos += 1; self.show_current()

    def on_undo(self, b):
        if self.history:
            last_pos = self.history.pop()
            self.my_labels.pop(self.items[self.todo_indices[last_pos]]['id'], None)
            save_labels(self.csv_path, self.my_labels)
            self.current_pos = last_pos; self.show_current()

    def on_quit(self, b):
        self.main_box.close()

    def show_current(self):
        with self.out_status:
            clear_output()
            if self.current_pos >= len(self.todo_indices):
                print(f"All labeled ({len(self.my_labels)}/{len(self.items)}).")
                return
            item = self.items[self.todo_indices[self.current_pos]]
            print(f"[{self.current_pos + 1}/{len(self.todo_indices)}] {item['id']} | {'OFFSIDE' if item['y'] == 1 else 'ONSIDE'}")
        with self.out_image:
            clear_output(wait=True)
            if self.current_pos < len(self.todo_indices):
                item = self.items[self.todo_indices[self.current_pos]]
                img, _ = load_and_align_image(item['path'], is_left=item['is_left'])
                fig, ax = plt.subplots(figsize=(11, 6)); ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis('off')
                plt.show(); plt.close(fig)


annotator = KitAnnotator(items, todo_indices, my_labels, my_csv)


In [ ]:
import csv, io, contextlib
import pandas as pd
from tqdm.auto import tqdm

FEATURE_NAMES = ['diff_x', 'second_attacker_diff', 'y_diff', 'scale_ratio',
                 'defensive_line_spread', 'attacker_x_norm', 'ball_x_diff']


def load_all_kits():
    kits, p = {}, BASE_DIR / 'attacking_kits.csv'
    if p.exists():
        with open(p, encoding='utf-8') as f:
            for row in csv.DictReader(f):
                if row.get('kit'):
                    kits[row['image_id']] = row['kit']
    return kits


def kit_arg(kit):
    if not kit or not isinstance(kit, str) or kit == 'auto':
        return None
    return tuple(int(v) for v in kit.split(',')) if ',' in kit else kit


def compute_features(img, ball_pos, attackers, defenders, goalkeeper):
    h, w = img.shape[:2]
    att = sorted(attackers, key=lambda p: p['max_x'], reverse=True)
    a1, a2 = att[0], (att[1] if len(att) > 1 else att[0])
    opp = sorted(defenders + ([goalkeeper] if goalkeeper is not None else []), key=lambda p: p['max_x'], reverse=True)
    k = 1 if (goalkeeper is not None and len(opp) >= 2) else 0
    d1, d2 = opp[k], (opp[k + 1] if len(opp) > k + 1 else opp[k])

    feats = np.array([
        (a1['max_x'] - d1['max_x']) / w, (a2['max_x'] - d1['max_x']) / w,
        (a1['max_point'][1] - d1['max_point'][1]) / h, a1['box_height'] / (d1['box_height'] + 1e-6),
        (d1['max_x'] - d2['max_x']) / w, a1['max_x'] / w,
        ((a1['max_x'] - ball_pos[0]) / w) if ball_pos is not None else 0.0,
    ], dtype=np.float32)
    return feats, {'n_att': len(attackers), 'n_def': len(defenders), 'gk': goalkeeper is not None, 'ball': ball_pos is not None}


items, kits = list_dataset(), load_all_kits()
X_list, y_list, names, rows = [], [], [], []
for it in tqdm(items):
    img, _ = load_and_align_image(it['path'], is_left=it['is_left'])
    mask, _ = get_pitch_mask(img)
    with contextlib.redirect_stdout(io.StringIO()):
        players, ball_pos = detect_players_and_ball(img, mask, det_model, pose_model)
        if len(players) < 4:
            continue
        attackers, defenders, goalkeeper, _ = cluster_teams(players, img, attacking_kit=kit_arg(kits.get(it['id'])))
    if not attackers or not defenders:
        continue
    feats, info = compute_features(img, ball_pos, attackers, defenders, goalkeeper)
    X_list.append(feats); y_list.append(it['y']); names.append(it['id'])
    rows.append({'image_id': it['id'], 'y': it['y'], 'kit': kits.get(it['id'], 'auto'),
                 **dict(zip(FEATURE_NAMES, feats.tolist())), **info, 'rule_pred': int(feats[0] > 0)})

X, y = np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.int64)
np.savez(BASE_DIR / 'features.npz', X=X, y=y, names=np.array(names), feature_names=np.array(FEATURE_NAMES))
pd.DataFrame(rows).to_csv(BASE_DIR / 'features_debug.csv', index=False, encoding='utf-8')
print(f"Baseline (diff_x > 0) Accuracy: {float(((X[:, 0] > 0).astype(int) == y).mean()) * 100:.1f}%")


In [ ]:
import pandas as pd

dbg = pd.read_csv(BASE_DIR / 'features_debug.csv')
wrong = dbg[dbg['rule_pred'] != dbg['y']].reset_index(drop=True)
print(f"Incorrect predictions: {len(wrong)} / {len(dbg)}")
